```mermaid
graph LR
 日本語+Pythonのコーパス作成 --> トークナイザーの学習
 トークナイザーの学習 --> GPTモデルの設計
  GPTモデルの設計 --> 学習ループの実装
  学習ループの実装 --> ファインチューニング
  ファインチューニング --> 推論
```

| Step | Name |Description |
| ---- | ----------- |----------- |
| 1 | 日本語+Pythonのコーパス作成 | (1)日本語(20~200MB) --> Wikipedia, 青空文庫, ニュース系コーパス, (2)Python(20~200MB) --> GitHub, Kaggle Notebooks |
|2 | トークナイザーの学習 | 語彙数(16k~32k), 日本語(BPE < Unigram), Python code(インデントや記号を分離しすぎない) |
|3| GPTモデルの設計 | Embedding(token embedding, position embedding), Transformer Block(Multi-Head Attention, MLP, Layer Normalization, Residual), Language Model Heads(Softmax) |
|4| 学習ループの実装 | batch size(16~48), sequence length(256~512), learning time(1h~1day) |
|5| ファインチューニング | Kaggle Notebooks, Python code, Python Q&A |
|6| 推論 ||

In [8]:
from datasets import load_dataset
import pandas as pd

In [3]:
# Japanese datasets 
ds_wiki = load_dataset("mini97/filtered_japanese-wikipedia") 
ds_aozora = load_dataset("globis-university/aozorabunko-clean") 

# Python datasets 
ds_python = load_dataset("Arjun-G-Ravi/Python-codes")

In [10]:
print(type(ds_wiki))
print(type(ds_aozora))
print(type(ds_python))

<class 'datasets.dataset_dict.DatasetDict'>
<class 'datasets.dataset_dict.DatasetDict'>
<class 'datasets.dataset_dict.DatasetDict'>


In [12]:
ds_wiki.column_names

{'train': ['text',
  'meta',
  'original',
  'mean_paragraph_length',
  'num_paragraphs',
  'short_paragraph_count',
  'total_length',
  'main_language_ratio']}

In [13]:
ds_aozora.column_names

{'train': ['text', 'footnote', 'meta']}

In [14]:
ds_python.column_names

{'train': ['code', 'question']}

In [15]:
ds_wiki["train"][0]

{'text': '『勝つか死ぬか』はHBO(日本ではスター・チャンネルが放送)のファンタジー・ドラマ・シリーズである『ゲーム・オブ・スローンズ』の第1章『七王国戦記』の第7話である。プロデューサーでもあるデイヴィッド・ベニオフ と D・B・ワイスが脚本を書き、 ダニエル・ミナハンが監督した。\n\n本エピソードでは、七王国の政治バランスの崩壊がさらに進み、ロバート王が狩りで外出している間に、エダードが発見した事実をサーセイに明らかにする。タイトルはサーセイの言葉「王座争奪戦では勝つか死ぬかです。妥協点はありません。」の引用である。このキャッチフレーズは原作本およびTVシリーズのプロモーションで多用されたものである。\n\nあらすじ\n\nラニスターの陣\nタイウィン・ラニスター公(チャールズ・ダンス)は牡鹿の皮をはぎながら、息子のジェイミー(ニコライ・コスター＝ワルドー)と話す。スターク家との争いを起こしたことを責めながらも、ラニスター家が七王国を統治する王朝を築く絶好の機会であるとタイウィンは信じる。キャトリンがティリオンを逮捕したことへの復讐として、軍の半分をジェイミーに与えて、タリー家の本拠でレディ・キャトリンの生家であるリヴァーランを攻めさせる。\n\nウィンターフェル\n捕えられた〈野人〉のオシャ(ナタリア・テナ)はスターク家の召使となり、シオン(アルフィー・アレン)に、もしもシオンの故郷の鉄諸島で捕えられていたならもっとひどい目に会っていたはずだとからかわれる。オシャに迫ろうとしたシオンを目撃したメイスター・ルイーウィンはシオンを追い払い、なぜ〈野人〉たちが〈壁〉の南に逃げてくるのかとオシャに問う。オシャは数千年の眠りから覚めたホワイト・ウォーカーから逃げてきたのだと言い、七王国のすべての軍は北に進軍してこの脅威に対処すべきだと言う。',
 'meta': {'id': '2969837',
  'title': '勝つか死ぬか',
  'url': 'https://ja.wikipedia.org/wiki/%E5%8B%9D%E3%81%A4%E3%81%8B%E6%AD%BB%E3%81%AC%E3%81%8B'},
 'original': '『勝つか死ぬか』はHBO(日本ではスター・チャンネルが放送)のファンタジー・ドラマ・シリーズである『